## pool config

In [ ]:
import os
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
from GetPoolData import get_pool_data_bigquery, get_pool_data_flipside
import config

# Set Google credentials for BigQuery
os.environ["GOOGLE_APPLICATION_CREDENTIALS"] = config.GOOGLE_SERVICE_AUTH_JSON

# Create data directory if it doesn't exist
if not os.path.exists('./data'):
    os.makedirs('./data')
    print("Created data directory")
else:
    print("Data directory already exists")

# Define your pools here
pools_config = [
    {
        'address': '0xcbcc3cbad991ec59204be2963b4a87951e4d292b',  
        'decimals_0': 18,  # VITA
        'decimals_1': 18,  # WETH
        'filename': 'vita_weth_1',
        'network': 'mainnet',
        'description': 'VITA-WETH 1% pool'
    },
    {
        'address': '0x6c063a6e8cd45869b5eb75291e65a3de298f3aa8',  
        'decimals_0': 18,   # WETH 
        'decimals_1': 18,  # INJ
        'filename': 'weth_inj_03',
        'network': 'mainnet',
        'description': 'WETH-INJ 0.3% pool'
    },
    {
        'address': '0x4628a0a564debfc8798eb55db5c91f2200486c24',  
        'decimals_0': 18,   # RNDR      
        'decimals_1': 18,  # WETH
        'filename': 'rndr_weth_03',
        'network': 'mainnet',
        'description': 'RNDR-WETH 0.3% pool'
    },
    {
        'address': '0x77E06c9eCCf2E797fd462A92B6D7642EF85b0A44',  
        'decimals_0': 9,   # WTAO      
        'decimals_1': 18,  # WETH
        'filename': 'wtao_weth_1',
        'network': 'mainnet',
        'description': 'WTAO-WETH 1% pool'
    },
    {
        'address': '0xc8219b876753A85025156b22176c2eDEA17aAC53',  
        'decimals_0': 18,   # MORPHO      
        'decimals_1': 18,  # WETH
        'filename': 'morpho_weth_03',
        'network': 'mainnet',
        'description': 'MORPHO-WETH 0.3% pool'
    },
    {
        'address': '0xa6Cc3C2531FdaA6Ae1A3CA84c2855806728693e8',  
        'decimals_0': 18,   # LINK      
        'decimals_1': 18,  # WETH
        'filename': 'link_weth_03',
        'network': 'mainnet',
        'description': 'RNDR-WETH 0.3% pool'
    }
]

# Date range for data download
date_begin = "2025-01-01"
date_end = "2025-01-07"

# Display configuration
print(f"Date range: {date_begin} to {date_end}")
print(f"Number of pools: {len(pools_config)}")
print("\nPool configurations:")
for i, pool in enumerate(pools_config):
    print(f"{i+1}. {pool['description']}")
    print(f"   Address: {pool['address']}")
    print(f"   Decimals: {pool['decimals_0']} / {pool['decimals_1']}")
    print(f"   Network: {pool['network']}")
    print(f"   Filename: {pool['filename']}")
    print()

Data directory already exists
Date range: 2025-01-01 to 2025-01-07
Number of pools: 3

Pool configurations:
1. VITA-WETH 1% pool
   Address: 0xcbcc3cbad991ec59204be2963b4a87951e4d292b
   Decimals: 18 / 18
   Network: mainnet
   Filename: vita_weth_1

2. ETH-INJ 0.3% pool
   Address: 0x6c063a6e8cd45869b5eb75291e65a3de298f3aa8
   Decimals: 18 / 18
   Network: mainnet
   Filename: eth_inj_03

3. RNDR-ETH 0.3% pool
   Address: 0x4628a0a564debfc8798eb55db5c91f2200486c24
   Decimals: 18 / 18
   Network: mainnet
   Filename: rndr_weth_03



In [2]:
def download_pool_data_bigquery(pool_config, date_begin, date_end, force_download=False):
    """
    Download pool data using BigQuery method.
    
    Args:
        pool_config: Dictionary containing pool configuration
        date_begin: Start date for data
        date_end: End date for data
        force_download: If True, download even if file exists
    
    Returns:
        DataFrame with pool data or None if error
    """
    try:
        filename = f"./data/{pool_config['filename']}_bigquery.csv"
        
        # Check if file already exists and force_download is False
        if not force_download and os.path.exists(filename):
            print(f"Loading existing data for {pool_config['description']} from {filename}")
            data = pd.read_csv(filename, index_col='block_date', parse_dates=True)
            return data
        
        print(f"Downloading data for {pool_config['description']} using BigQuery...")
        
        # Download data using BigQuery
        data = get_pool_data_bigquery(
            pool_config['address'],
            date_begin,
            date_end,
            pool_config['decimals_0'],
            pool_config['decimals_1'],
            network=pool_config['network']
        )
        
        # Save to CSV
        data.to_csv(filename)
        print(f"Saved data to {filename}")
        print(f"Data shape: {data.shape}")
        print(f"Date range: {data.index.min()} to {data.index.max()}")
        
        return data
        
    except Exception as e:
        print(f"Error downloading data for {pool_config['description']}: {str(e)}")
        return None

def download_pool_data_flipside(pool_config, flipside_queries, force_download=False):
    """
    Download pool data using Flipside method.
    
    Args:
        pool_config: Dictionary containing pool configuration
        flipside_queries: List of Flipside query URLs
        force_download: If True, download even if file exists
    
    Returns:
        DataFrame with pool data or None if error
    """
    try:
        filename = f"./data/{pool_config['filename']}_flipside.csv"
        
        # Check if file already exists and force_download is False
        if not force_download and os.path.exists(filename):
            print(f"Loading existing data for {pool_config['description']} from {filename}")
            data = pd.read_csv(filename, index_col='time_pd', parse_dates=True)
            return data
        
        print(f"Downloading data for {pool_config['description']} using Flipside...")
        
        # Download data using Flipside
        data = get_pool_data_flipside(
            pool_config['address'],
            flipside_queries,
            pool_config['filename']
        )
        
        # Save to CSV
        data.to_csv(filename)
        print(f"Saved data to {filename}")
        print(f"Data shape: {data.shape}")
        print(f"Date range: {data.index.min()} to {data.index.max()}")
        
        return data
        
    except Exception as e:
        print(f"Error downloading data for {pool_config['description']}: {str(e)}")
        return None

In [3]:
# Download data for all pools using BigQuery
print("Starting BigQuery data download for all pools...")
print("=" * 60)

downloaded_data = {}

for i, pool in enumerate(pools_config):
    print(f"\nProcessing pool {i+1}/{len(pools_config)}: {pool['description']}")
    print("-" * 40)
    
    # Download data
    data = download_pool_data_bigquery(pool, date_begin, date_end)
    
    if data is not None:
        downloaded_data[pool['filename']] = data
        print(f"✓ Successfully downloaded data for {pool['description']}")
    else:
        print(f"✗ Failed to download data for {pool['description']}")

print("\n" + "=" * 60)
print(f"Download completed. Successfully downloaded {len(downloaded_data)} out of {len(pools_config)} pools.")

Starting BigQuery data download for all pools...

Processing pool 1/3: VITA-WETH 1% pool
----------------------------------------
Error downloading data for VITA-WETH 1% pool: Please install the 'db-dtypes' package to use this function.
✗ Failed to download data for VITA-WETH 1% pool

Processing pool 2/3: ETH-INJ 0.3% pool
----------------------------------------
Error downloading data for ETH-INJ 0.3% pool: Please install the 'db-dtypes' package to use this function.
✗ Failed to download data for ETH-INJ 0.3% pool

Processing pool 3/3: RNDR-ETH 0.3% pool
----------------------------------------
Error downloading data for RNDR-ETH 0.3% pool: Please install the 'db-dtypes' package to use this function.
✗ Failed to download data for RNDR-ETH 0.3% pool

Download completed. Successfully downloaded 0 out of 3 pools.


In [ ]:
# Display summary of downloaded data
print("Data Summary:")
print("=" * 60)

for filename, data in downloaded_data.items():
    print(f"\n{filename}:")
    print(f"  Shape: {data.shape}")
    print(f"  Date range: {data.index.min()} to {data.index.max()}")
    print(f"  Columns: {list(data.columns)}")
    print(f"  Memory usage: {data.memory_usage(deep=True).sum() / 1024**2:.2f} MB")
    
    # Show sample data
    print(f"  Sample data:")
    print(data.head(3))
    print()

In [ ]:
# Data quality checks
print("Data Quality Checks:")
print("=" * 60)

for filename, data in downloaded_data.items():
    print(f"\n{filename}:")
    
    # Check for missing values
    missing_values = data.isnull().sum()
    if missing_values.sum() > 0:
        print(f"  Missing values:")
        for col, missing in missing_values[missing_values > 0].items():
            print(f"    {col}: {missing} ({missing/len(data)*100:.2f}%)")
    else:
        print(f"  ✓ No missing values")
    
    # Check for duplicate timestamps
    duplicates = data.index.duplicated().sum()
    if duplicates > 0:
        print(f"  ⚠ {duplicates} duplicate timestamps")
    else:
        print(f"  ✓ No duplicate timestamps")
    
    # Check price data
    if 'quotePrice' in data.columns:
        price_stats = data['quotePrice'].describe()
        print(f"  Price statistics:")
        print(f"    Min: {price_stats['min']:.6f}")
        print(f"    Max: {price_stats['max']:.6f}")
        print(f"    Mean: {price_stats['mean']:.6f}")
        print(f"    Std: {price_stats['std']:.6f}")
    
    # Check for negative prices (shouldn't happen)
    if 'quotePrice' in data.columns:
        negative_prices = (data['quotePrice'] < 0).sum()
        if negative_prices > 0:
            print(f"  ⚠ {negative_prices} negative prices found")
        else:
            print(f"  ✓ No negative prices")
    
    print()